# Deepwater Subsea Well Control Case Study
## Modeling Choke Line Friction, Dynamic MAASP, and Riser Disconnect in 5,000 ft Water Depth

### Executive Summary
In deepwater drilling operations utilizing a subsea BOP stack, the presence of long, small-diameter choke lines between the seabed and surface introduces substantial frictional backpressure during circulating well kills. 

This case study demonstrates:
1. **Choke Line Friction (CLF)** measurement and its direct impact on annular backpressure.
2. **Dynamic MAASP Reduction:** Why the allowable casing gauge reading must decrease by the exact value of CLF to protect the casing shoe from hydraulic fracturing.
3. **Choke Pressure Protocol during Pump Ramp-Up:** Holding surface casing at $SICP - CLF$ to prevent bottomhole pressure spikes.
4. **Riser Margin Evaluation:** Quantifying whether the well will remain overbalanced or become underbalanced if the marine riser emergency-disconnects, replacing drilling mud above the seabed with a seawater hydrostatic head.

In [ ]:
from killsheet.subsea import (
    choke_line_capacity,
    choke_line_volume,
    choke_line_strokes,
    dynamic_maasp,
    choke_pressure_bringing_pump_up,
    riser_margin_mud_weight,
    riser_disconnect_bhp,
    subsea_kill_summary,
)
from killsheet.formulas import kill_mud_weight, maasp

# Well Geometry & Fluid Conditions
water_depth_ft = 5000
air_gap_ft = 85
tvd_ft = 18500
shoe_tvd_ft = 13200
omw_ppg = 13.6
lot_emw_ppg = 16.2

# Shut-In Kick Data
sidpp_psi = 750
sicp_psi = 1150
scr_pressure_psi = 850          # At 30 SPM
pump_output_bbl_per_stk = 0.108

# Choke Line Configuration
choke_line_id_in = 3.0
choke_line_length_ft = 5250     # Water depth + air gap + stack/moonpool iron
clf_psi = 420                   # Measured CLF at 30 SPM kill rate

### 1. Choke Line Displacement & Volumetrics
Because the choke line has an ID of 3.0 inches over nearly a mile of length, it contains a critical fluid volume that must be accounted for during kill mud displacement.

In [ ]:
cl_cap = choke_line_capacity(choke_line_id_in)
cl_vol = choke_line_volume(choke_line_id_in, choke_line_length_ft)
cl_stk = choke_line_strokes(cl_vol, pump_output_bbl_per_stk)

print(f"Choke Line Capacity: {cl_cap:.5f} bbl/ft")
print(f"Choke Line Total Volume: {cl_vol:.1f} bbl")
print(f"Strokes to Displace Choke Line: {cl_stk:,} strokes")

### 2. Kill Mud Weight & Static vs. Dynamic MAASP
The maximum allowable annular surface pressure (MAASP) under static shut-in conditions is governed by the casing shoe leak-off test. However, when circulating, choke line friction builds up, adding backpressure directly to the wellbore below the subsea BOP stack.

In [ ]:
kmw = kill_mud_weight(sidpp_psi, tvd_ft, omw_ppg)

# Static MAASP = (LOT - OMW) x 0.052 x Shoe TVD
static_maasp_psi = maasp(lot_emw_ppg, omw_ppg, shoe_tvd_ft)

# Dynamic MAASP = Static MAASP - CLF
dyn_maasp_psi = dynamic_maasp(static_maasp_psi, clf_psi)

print(f"Kill Mud Weight (KMW): {kmw:.1f} ppg (rounded UP per IADC WellSharp)")
print(f"Static MAASP (Well Shut-In): {static_maasp_psi:,} psi")
print(f"Choke Line Friction (CLF):   {clf_psi:,} psi")
print(f"Dynamic MAASP (Circulating): {dyn_maasp_psi:,} psi")

### 3. Surface Choke Management on Pump Start-Up
On a surface stack, the choke operator holds casing pressure constant at $SICP$ while bringing the pump to kill speed. 

On a subsea stack, if the operator holds $SICP$ (1,150 psi), the formation feels $SICP + CLF = 1,150 + 420 = 1,570\text{ psi}$, which overpressures the open hole. Instead, the choke must be opened to drop the surface reading to $SICP - CLF$.

In [ ]:
target_choke_pressure = choke_pressure_bringing_pump_up(sicp_psi, clf_psi)
print(f"Initial SICP reading:             {sicp_psi:,} psi")
print(f"Target choke gauge during ramp:   {target_choke_pressure:,} psi (SICP - CLF)")

### 4. Riser Disconnect & Overbalance Analysis
If harsh weather or dynamically positioned (DP) vessel drive-off forces an emergency disconnect, the marine riser unlatches at the LMRP. The drilling mud column from seabed to rig floor is replaced by an unconfined column of seawater.

In [ ]:
mw_rm = riser_margin_mud_weight(omw_ppg, tvd_ft, water_depth_ft, air_gap_ft)
post_disc_bhp = riser_disconnect_bhp(omw_ppg, tvd_ft, water_depth_ft, air_gap_ft)
formation_p = sidpp_psi + 0.052 * omw_ppg * tvd_ft
margin = post_disc_bhp - formation_p

print(f"Riser Margin Mud Weight Required: {mw_rm:.2f} ppg (+{mw_rm - omw_ppg:.2f} ppg increase)")
print(f"Formation Pressure:                {formation_p:,.0f} psi")
print(f"Post-Disconnect BHP:               {post_disc_bhp:,.0f} psi")
if margin >= 0:
    print(f"Status: OVERBALANCED by {margin:,.0f} psi")
else:
    print(f"Status: CRITICAL UNDERBALANCE by {abs(margin):,.0f} psi - Well will flow at the seabed!")